# Twisted bilayer Boron Nitride at 7.34° on a commensurate lattice.

## 0. Introduction.

This notebook demonstrates how to generate a twisted interface between two materials using commensurate lattices. The example uses hexagonal boron nitride (h-BN) as both the film and substrate materials and builds the periodic moiré cells of configurations α and β that the paper computes, at 7.34° (244 atoms) by default. The notebook uses the `create_interface_commensurate` function which first creates a slab from the material and then performs commensurate lattice matching to find valid supercells for the target twist angle. The algorithm searches for the supercell matrices that give the target twist angle. The generated interfaces are visualized and analyzed to determine the number of atoms and the interlayer distance.

> **Lede Xian, Dante M. Kennes, Nicolas Tancogne-Dejean, Massimo Altarelli, and Angel Rubio**
> Multiflat Bands and Strong Correlations in Twisted Bilayer Boron Nitride: Doping-Induced Correlated Insulator and Superconductor. Nano Letters, 19, 4934. 2019.
> [https://doi.org/10.1021/acs.nanolett.9b00986](https://doi.org/10.1021/acs.nanolett.9b00986)

The twisted h-BN bilayer is shown in the following Figure 1 from the article: configurations α (a) and β (b), and the width of the top valence flat bands against twist angle (g).

<img src="https://github.com/Exabyte-io/documentation/raw/12617167278ae3523adc028583b21ea4e8ebd197/images/tutorials/materials/interfaces/twisted-bilayer-boron-nitride/tbbn-paper-image.png" alt="Twisted bilayer h-BN" width="800"/>

## 1. Prepare the Environment
### 1.1. Configurations from the article

The twist angles and the interlayer distance below are the ones of the article's band-width series: flat layers,
3.23 Å apart, the optimized interlayer distance of the untwisted bilayer. The article's two configurations, α and β,
share one moiré cell and differ in twist by 60°: in α the B-over-B and N-over-N regions sit at different sites of the
cell, in β at the same site.

The names below are how the [band structure notebook](interface_bilayer_twisted_commensurate_lattices_boron_nitride_SIMULATION.ipynb)
finds the materials created here, so changing one means changing it there too.

In [ ]:
# Uncomment a line to build that configuration as well: α is twisted by 60° + θ, β by θ. The angles are those of
# Xian et al. 2019 Fig. 1g; smaller angles have larger cells (6.01°: 364 atoms, 5.08°: 508, 3.89°: 868, 2.64°: 1876).
INTERFACE_PARAMETERS = [
    {"name": "h-BN bilayer 7.34deg alpha d3.23", "angle": 60 + 7.34},
    {"name": "h-BN bilayer 7.34deg beta d3.23", "angle": 7.34},
    # {"name": "h-BN bilayer 6.01deg alpha d3.23", "angle": 60 + 6.01},
    # {"name": "h-BN bilayer 6.01deg beta d3.23", "angle": 6.01},
    # {"name": "h-BN bilayer 5.08deg alpha d3.23", "angle": 60 + 5.08},
    # {"name": "h-BN bilayer 5.08deg beta d3.23", "angle": 5.08},
    # {"name": "h-BN bilayer 3.89deg alpha d3.23", "angle": 60 + 3.89},
    # {"name": "h-BN bilayer 3.89deg beta d3.23", "angle": 3.89},
    # {"name": "h-BN bilayer 2.64deg alpha d3.23", "angle": 60 + 2.64},
    # {"name": "h-BN bilayer 2.64deg beta d3.23", "angle": 2.64},
]
INTERFACE_DISTANCE = 3.23  # Å, the article's

### 1.2. Set interface creation and visualization parameters

In [ ]:
VACUUM = 20.0  # Å of empty space above the top layer; paper: not stated

# Visualization parameters
SHOW_INTERMEDIATE_STEPS = True
VISUALIZE_REPETITIONS = [3, 3, 1]

### 1.3. Install packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.4. Get input material
We'll use the h-BN monolayer from Standata.


In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

# paper: a = 2.512 Å; ours Standata's h-BN monolayer as is
material = Material.create(Materials.get_by_name_and_categories("BN", "2D"))
print(f"{material.name}: {material.basis.number_of_atoms} atoms, a = {material.lattice.a:.3f} Å")

if SHOW_INTERMEDIATE_STEPS:
    visualize_materials(material, repetitions=VISUALIZE_REPETITIONS)
    visualize_materials(material, repetitions=VISUALIZE_REPETITIONS, rotation="-90x")

## 2. Generate the Twisted Interfaces

In [ ]:
from mat3ra.made.tools.analyze.other import get_average_interlayer_distance
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.convert.interface_parts_enum import InterfacePartsEnum
from mat3ra.made.tools.helpers import create_interface_commensurate, create_supercell

interfaces = []
for parameters in INTERFACE_PARAMETERS:
    interface = create_interface_commensurate(
        material=material,
        target_angle=parameters["angle"],
        gap=INTERFACE_DISTANCE,
        # the builder adds the gap above the top layer as well
        vacuum=VACUUM - INTERFACE_DISTANCE,
    )
    # The symbolic K point of the band path assumes the 120° hexagonal cell; the builder returns the 60° one.
    interface = create_supercell(interface, supercell_matrix=[[1, 0, 0], [-1, 1, 0], [0, 0, 1]])
    interface.lattice.type = "HEX"
    interface.name = parameters["name"]
    interfaces.append(interface)

    interlayer_distance = get_average_interlayer_distance(
        interface, InterfacePartsEnum.SUBSTRATE.value, InterfacePartsEnum.FILM.value)
    print(f"{interface.name}: {interface.basis.number_of_atoms} atoms ({to_ase(interface).get_chemical_formula()}), "
          f"a = {interface.lattice.a:.3f} Å, gamma = {interface.lattice.gamma:.1f}°, "
          f"interlayer distance {interlayer_distance:.3f} Å, target twist {parameters['angle']:.2f}°")

## 3. Preview the materials

In [ ]:
titled_interfaces = [{"material": interface, "title": interface.name} for interface in interfaces]
visualize_materials(titled_interfaces, repetitions=VISUALIZE_REPETITIONS)
visualize_materials(titled_interfaces, repetitions=VISUALIZE_REPETITIONS, rotation="-90x")

## 4. Save the materials

Each material is written to the `uploads` folder under its name, which is how the
[band structure notebook](interface_bilayer_twisted_commensurate_lattices_boron_nitride_SIMULATION.ipynb)
picks it up.

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

set_materials(interfaces)
for interface in interfaces:
    download_content_to_file(interface.to_json(), f"{interface.name}.json")

## References

[1] L. Xian, D. M. Kennes, N. Tancogne-Dejean, M. Altarelli and A. Rubio, "Multiflat Bands and Strong Correlations in Twisted Bilayer Boron Nitride: Doping-Induced Correlated Insulator and Superconductor", Nano Lett. 19, 4934 (2019). https://doi.org/10.1021/acs.nanolett.9b00986